# 04 · Segmentación y estimación de pose

La oferta pide "conocimientos de detección, segmentación y pose estimation" y valora "RTMPose, ViTPose, SAM".
Objetivos:
- Distinguir los tipos de segmentación y saber cómo segmenta YOLO11-seg (prototipos).
- Entender **SAM / SAM2** (segmentación *promptable*) y para qué se usa (anotación, tracking de máscaras).
- Pose: **top-down vs bottom-up vs one-stage**, heatmaps vs regresión vs SimCC, y la métrica **OKS** ✍️.
- Sacar información útil de la pose (ángulos, punto de apoyo en el suelo).

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
ROOT = ROOT or Path.cwd()   # fuera del repo (Colab): los datos se guardan en ./data
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
from pathlib import Path
import time
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
cap = cv2.VideoCapture(str(DATA / "people.mp4"))
ok, frame = cap.read()
cap.release()

def show(img, title="", size=(12, 7)):
    plt.figure(figsize=size); plt.imshow(img[:, :, ::-1]); plt.title(title); plt.axis("off"); plt.show()

## 1. Tipos de segmentación

| Tipo | Qué da | Ejemplos |
|---|---|---|
| **Semántica** | clase por píxel; todas las personas = misma "mancha" | U-Net, DeepLabv3+, SegFormer |
| **Instancias** | una máscara por objeto | Mask R-CNN, **YOLO11-seg**, Mask2Former |
| **Panóptica** | semántica (fondo: césped, gradas) + instancias (jugadores) | Mask2Former, OneFormer |
| **Promptable** | máscara del objeto que le señales (punto, caja, texto) | **SAM, SAM2** |

Métricas: **mask IoU**, mAP de máscaras (como el de cajas pero con IoU de máscaras), **mIoU** en semántica,
Dice = 2|A∩B|/(|A|+|B|) (habitual en médica).

### Cómo segmenta YOLO11-seg (estilo YOLACT)
Además de la cabeza de detección, una rama genera **32 "prototipos"** (máscaras base a 1/4 de resolución,
160×160) compartidos por toda la imagen. Para cada detección se predicen **32 coeficientes**;
la máscara = `sigmoid(coefs · prototipos)`, recortada a la caja. Es barato: el coste casi no crece con el
nº de objetos.

In [ ]:
seg = YOLO(str(DATA / "yolo11n-seg.pt"))
r = seg(frame, classes=[0], device=DEVICE, verbose=False)[0]
print("detecciones:", len(r.boxes), "| máscaras:", tuple(r.masks.data.shape), "(N, H, W) a resolución de entrada")
print("polígono de la 1ª máscara (px originales):", r.masks.xy[0].shape)
show(r.plot(boxes=False), "YOLO11n-seg")

In [ ]:
def mask_iou(m1, m2):
    m1, m2 = m1.astype(bool), m2.astype(bool)
    return (m1 & m2).sum() / max((m1 | m2).sum(), 1)

# máscara binaria a tamaño original desde el polígono (así se guarda en COCO y en YOLO-seg)
H, W = frame.shape[:2]
m0 = np.zeros((H, W), np.uint8)
cv2.fillPoly(m0, [r.masks.xy[0].astype(np.int32)], 1)
x1, y1, x2, y2 = r.boxes.xyxy[0].cpu().numpy().astype(int)
box_mask = np.zeros_like(m0); box_mask[y1:y2, x1:x2] = 1
print(f"IoU máscara vs su propia caja = {mask_iou(m0, box_mask):.2f}  (una persona ocupa ~50-60% de su caja)")

**¿Por qué segmentar si ya tengo cajas?** En tracking/deporte:
- Recortar **solo los píxeles del jugador** para el color de camiseta o el ReID (la caja incluye césped y otros jugadores).
- Oclusiones: dos cajas solapadas pueden tener máscaras disjuntas → asociación más fiable.
- Mejor punto de contacto con el suelo (el píxel más bajo de la máscara).

## 2. SAM y SAM2: *Segment Anything*

**SAM** (Meta, 2023): modelo fundacional entrenado con 1.100 M de máscaras (SA-1B).
- **Image encoder** pesado (ViT, MAE) → se ejecuta **una vez** por imagen.
- **Prompt encoder**: puntos (positivos/negativos), cajas, máscaras.
- **Mask decoder** ligero → **milisegundos por prompt** → interactivo.
- Sin clases: segmenta "lo que señales". Por eso se combina con un detector: **YOLO da cajas → SAM da máscaras**.

**SAM2** (2024): extiende a **vídeo** con un **memory bank**: le das un prompt en un frame y propaga la máscara
del objeto por todo el vídeo, recuperándose de oclusiones. Es, en la práctica, un **tracker de máscaras**
(VOS, *video object segmentation*). **SAM 3** (2025) añade prompts de concepto por texto ("jugador con camiseta
roja") detectando y siguiendo todas las instancias.

Usos en el puesto:
1. **Anotación asistida**: CVAT y Roboflow integran SAM → un clic = máscara. Acelera muchísimo crear datasets.
2. Generar máscaras a partir de cajas existentes (convertir un dataset de detección en uno de segmentación).
3. Seguimiento de objetos difíciles en vídeo (offline, no es tiempo real con los modelos grandes).
Variantes rápidas: MobileSAM, EfficientSAM, FastSAM, EdgeTAM.

In [ ]:
from ultralytics import SAM

sam = SAM(str(DATA / "sam2.1_t.pt"))      # SAM2.1 tiny (~40 MB, se descarga sola)
boxes = r.boxes.xyxy.cpu().numpy()[:6]
t0 = time.perf_counter()
sr = sam(frame, bboxes=boxes, device=DEVICE, verbose=False)[0]
print(f"SAM2 con {len(boxes)} cajas como prompt: {time.perf_counter()-t0:.2f}s")
show(sr.plot(boxes=False), "SAM2.1-tiny con prompts de caja de YOLO")

In [ ]:
# Prompt de un punto: clic en el centro de una persona
cx, cy = ((boxes[0, :2] + boxes[0, 2:]) / 2).astype(int)
sp = sam(frame, points=[[cx, cy]], labels=[1], device=DEVICE, verbose=False)[0]
show(sp.plot(boxes=False), f"SAM2 con un clic en ({cx},{cy})")

## 3. Estimación de pose

Objetivo: localizar **K keypoints** por persona. COCO usa **17**: nariz, ojos, orejas, hombros, codos, muñecas,
caderas, rodillas, tobillos. Cada keypoint: `(x, y, visibilidad)` con v = 0 no etiquetado, 1 etiquetado pero
oculto, 2 visible.

### Tres enfoques
| Enfoque | Cómo | Pros / contras | Modelos |
|---|---|---|---|
| **Top-down** | detector de personas → recorte de cada una → modelo de pose por recorte | más preciso; coste **lineal en nº de personas**; depende del detector | **ViTPose**, **RTMPose**, HRNet |
| **Bottom-up** | detecta todos los keypoints de la imagen y luego los agrupa por persona | coste constante; agrupar en multitudes es difícil | OpenPose (PAFs), HigherHRNet |
| **One-stage** | la cabeza del detector predice caja + keypoints a la vez | rápido, coste ~constante; algo menos preciso | **YOLO11-pose**, RTMO |

### Cómo se representan los keypoints
- **Heatmaps**: un mapa por keypoint con una gaussiana en su posición; se toma el máximo. Precisos pero
  limitados por la resolución del mapa (cuantización) y costosos. (HRNet, ViTPose)
- **Regresión directa** de coordenadas: rápido, históricamente menos preciso (YOLO-pose, con loss basada en OKS).
- **SimCC** (RTMPose): clasifica la x y la y por separado como dos vectores 1D con resolución sub-píxel → precisión
  de heatmap con coste mucho menor. Por eso RTMPose es *real-time*.

**ViTPose**: un ViT "plano" (preentrenado con MAE) + decoder simple → SOTA, escala muy bien con el tamaño.
**RTMPose** (OpenMMLab/MMPose): backbone CSPNeXt + SimCC, pensado para despliegue (ONNX/TensorRT, móvil).

In [ ]:
pose = YOLO(str(DATA / "yolo11n-pose.pt"))
pr = pose(frame, device=DEVICE, verbose=False)[0]
kpts = pr.keypoints
print("keypoints:", tuple(kpts.data.shape), "= (personas, 17, [x, y, conf])")
show(pr.plot(), "YOLO11n-pose")

COCO_KPTS = ["nariz", "ojo_i", "ojo_d", "oreja_i", "oreja_d", "hombro_i", "hombro_d", "codo_i", "codo_d",
             "muñeca_i", "muñeca_d", "cadera_i", "cadera_d", "rodilla_i", "rodilla_d", "tobillo_i", "tobillo_d"]

### Sacar información de la pose
Ejemplos de lo que pedirían en análisis deportivo:
- **Ángulo de rodilla** (cadera–rodilla–tobillo) → ¿está corriendo, saltando, agachado?
- **Punto de apoyo**: el punto medio de los tobillos es mejor que el borde inferior de la caja para proyectar
  al plano del campo con la homografía (notebook 08), sobre todo si la caja está mal ajustada.
- Orientación del cuerpo (vector entre hombros), brazos levantados (muñeca por encima del hombro).

In [ ]:
def angle(a, b, c):
    """Ángulo en b (grados) formado por los segmentos b→a y b→c."""
    ba, bc = a - b, c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

K = kpts.data.cpu().numpy()
for i, kp in enumerate(K[:5]):
    hip, knee, ankle = kp[11, :2], kp[13, :2], kp[15, :2]
    ok_conf = kp[[11, 13, 15], 2].min() > 0.5
    foot = kp[[15, 16], :2].mean(0)
    bx = pr.boxes.xyxy[i].cpu().numpy()
    print(f"persona {i}: rodilla izq = {angle(hip, knee, ankle):5.1f}°{'' if ok_conf else ' (baja conf)'} | "
          f"pie (tobillos) y={foot[1]:.0f} vs borde caja y={bx[3]:.0f}")

## 4. ✍️ EJERCICIO: OKS (Object Keypoint Similarity)

Es el "IoU de la pose". Para una predicción y un GT:

$$ OKS = \frac{\sum_i \exp\left(-\frac{d_i^2}{2 s^2 k_i^2}\right)\,\delta(v_i>0)}{\sum_i \delta(v_i>0)} $$

- $d_i$: distancia euclídea entre keypoint predicho y GT.
- $s^2$: **área** del objeto (del GT) → normaliza por escala (un error de 5 px es grave en una persona pequeña).
- $k_i$: constante por keypoint (los ojos se anotan con precisión, las caderas no) → en COCO `sigmas` abajo, $k_i = 2\sigma_i$.
- Solo cuentan los keypoints etiquetados en el GT ($v_i > 0$).

Luego el **AP de pose** se calcula exactamente igual que el de cajas pero usando OKS en vez de IoU
(umbrales 0.50:0.95).

In [ ]:
COCO_SIGMAS = np.array([.26, .25, .25, .35, .35, .79, .79, .72, .72, .62, .62, 1.07, 1.07, .87, .87, .89, .89]) / 10

In [ ]:
# SOLUCIÓN
def oks(pred, gt, vis, area, sigmas=COCO_SIGMAS):
    k = 2 * sigmas
    d2 = ((pred - gt) ** 2).sum(1)
    e = np.exp(-d2 / (2 * area * k ** 2 + 1e-9))
    m = vis > 0
    return e[m].sum() / max(m.sum(), 1)

In [ ]:
gt = K[0, :, :2]; vis = (K[0, :, 2] > 0.5).astype(int) * 2
b = pr.boxes.xywh[0].cpu().numpy(); area = b[2] * b[3] * 0.53   # COCO usa área de la máscara ≈ 53% de la caja
for noise in [0, 2, 5, 10, 20]:
    print(f"ruido {noise:2d}px → OKS = {oks(gt + np.random.randn(17, 2) * noise, gt, vis, area):.3f}")

**Suavizado temporal de keypoints**: en vídeo los keypoints "tiemblan" (*jitter*). Se suavizan con un filtro de
Kalman por keypoint o con el **One Euro filter** (filtro paso-bajo adaptativo: suaviza mucho cuando el punto va
lento y poco cuando va rápido → poco retardo). Lo veremos con trayectorias en el notebook 09.